# Engagement 2: Mobile-money Agent Data Cleanup

## Data-quality audit

The audit records each detected issue by dimension, column, and affected-row count before analysis. Raw data is loaded with explicit date handling and the generator is the reproducible source of the deliberately injected issues.

In [1]:
import pandas as pd
from cleaning_pipeline import clean_agent_data

txns = pd.read_csv('agent_transactions.csv', keep_default_na=True)
agents = pd.read_csv('agents.csv')
print('Raw transaction rows:', len(txns))
print('Agent rows:', len(agents))
print('Raw types:', sorted(txns['txn_type'].dropna().unique()))
print('Raw amount missing:', txns['amount'].isna().sum())
clean_df, cleaning_log = clean_agent_data(txns, agents)
audit_table = pd.DataFrame(cleaning_log, columns=['issue', 'dimension', 'column', 'rows_affected'])
audit_table

Raw transaction rows: 612
Agent rows: 20
Raw types: ['DEPOSIT ', 'Deposit', 'Withdrawal', 'deposit', 'withdrawal']
Raw amount missing: 22


,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Non-standard transaction type formatting,Consistency,txn_type,376
2,Missing or invalid transaction types,Consistency,txn_type,0
3,Missing or non-numeric amounts,Completeness,amount,20
4,IQR outliers retained and flagged,Validity,amount,53
5,Non-ISO transaction date formatting,Consistency,txn_date,5
6,Invalid transaction dates,Validity,txn_date,0
7,Unknown agent IDs,Referential integrity,agent_id,7


## Cleaning decisions

Duplicate transaction IDs are reduced to one record per ID. Transaction types are stripped, lowercased, mapped to exactly `Deposit` or `Withdrawal`, and invalid or missing types would be excluded. Amounts are coerced to numeric; missing or invalid values are imputed with the median of valid amounts and marked in `amount_imputed`, preserving rows while making the treatment visible. Dates accept both hyphen and slash separators and are saved as ISO dates. IQR outliers are retained and flagged because an extreme value may be a genuine transaction requiring business investigation rather than automatic deletion. Unknown agents are retained after a left merge with a missing county so the governance issue remains countable.

## Post-merge validation

In [2]:
validation = pd.DataFrame({
    'measure': ['Raw transaction rows', 'Clean transaction rows', 'Unique txn_id values', 'Unknown-agent transactions', 'Duplicate txn_id values after cleaning'],
    'value': [len(txns), len(clean_df), clean_df['txn_id'].nunique(), clean_df['county'].isna().sum(), clean_df['txn_id'].duplicated().sum()]
})
validation

,measure,value
0,Raw transaction rows,612
1,Clean transaction rows,600
2,Unique txn_id values,600
3,Unknown-agent transactions,7
4,Duplicate txn_id values after cleaning,0


## County summary

In [3]:
county_summary = clean_df.groupby(['county', 'txn_type'], dropna=False).agg(
    total_value=('amount', 'sum'),
    number_of_transactions=('txn_id', 'nunique'),
    average_value=('amount', 'mean')
).reset_index()
county_summary

,county,txn_type,total_value,number_of_transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77050.5,27,2853.722222
2,Kisumu,Deposit,436420.5,74,5897.574324
3,Kisumu,Withdrawal,76374.5,37,2064.175676
4,Mombasa,Deposit,148986.0,72,2069.25
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543590.0,118,4606.694915
7,Nairobi,Withdrawal,162036.0,62,2613.483871
8,Nakuru,Deposit,184890.0,60,3081.5
9,Nakuru,Withdrawal,328594.5,39,8425.5


## Weekly transaction-value pivot

In [4]:
clean_df['week'] = pd.to_datetime(clean_df['txn_date']).dt.to_period('W').astype(str)
weekly_pivot = clean_df.pivot_table(
    index='week', columns='county', values='amount', aggfunc='sum', fill_value=0, dropna=False
)
weekly_pivot

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,NaN
week,,,,,,
2026-02-23/2026-03-01,0.0,1893.0,6759.0,7046.0,2743.5,0.0
2026-03-02/2026-03-08,18317.0,287933.0,285669.5,68967.0,56574.0,0.0
2026-03-09/2026-03-15,267771.5,23186.0,30477.5,52860.5,28730.0,0.0
2026-03-16/2026-03-22,22054.0,28125.0,25144.5,43107.5,39795.0,1639.0
2026-03-23/2026-03-29,11504.0,27077.0,17050.0,41373.5,18998.5,0.0
2026-03-30/2026-04-05,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06/2026-04-12,15030.0,15341.0,47092.5,56786.5,276609.0,9824.0
2026-04-13/2026-04-19,12265.0,29416.0,26730.0,56432.5,26152.5,0.0
2026-04-20/2026-04-26,24507.0,37439.0,16687.0,65187.0,30862.0,1877.0


In [5]:
clean_df.to_csv('outputs/agent_transactions_clean.csv', index=False)
print('Exported rows:', len(clean_df))
print('txn_type values:', sorted(clean_df['txn_type'].unique()))
print('ISO dates:', clean_df['txn_date'].str.fullmatch(r'\d{4}-\d{2}-\d{2}').all())

Exported rows: 600
txn_type values: ['Deposit', 'Withdrawal']
ISO dates: True


# Data Quality Report

## Uniqueness
The transaction extract contains repeated `txn_id` values. The pipeline keeps the first record for each ID, logs the number removed, and verifies that the cleaned output has no duplicate IDs.

## Consistency
Transaction-type spelling and case variants are standardised to exactly `Deposit` and `Withdrawal`. Missing or unrecognised values are identified before mapping and would be excluded rather than silently converted.

## Validity
Amounts are converted to numeric values and dates are parsed after normalising slash separators, then written in ISO `YYYY-MM-DD` form. IQR fences identify suspicious amounts; the KES 250,000 records are retained and flagged for operational investigation because deleting potentially genuine transactions would lose value.

## Completeness
Missing and non-numeric amounts are imputed with the median of valid amounts so valid transaction rows are not discarded. The `amount_imputed` flag preserves a transparent record of every replacement.

## Accuracy
The county and weekly summaries use the cleaned amount field, while outlier flags allow managers to compare reported totals with and without suspicious transactions. The pipeline is deterministic and records every treatment in its audit log.

## Referential integrity
A left join preserves every cleaned transaction and adds the agent county where the ID exists in the register. Unknown IDs remain visible with a missing county and are counted in post-merge validation rather than being silently dropped.

## Privacy
If customer phone numbers were present, they would be personal data under the Kenya Data Protection Act 2019. The business should collect them only for a defined lawful purpose, minimise access, encrypt them in transit and at rest, mask or pseudonymise them for analysis, restrict exports, and delete them when the retention purpose ends.